# AlphaFlow — Reproducible Research Notebook

This notebook reproduces every quantitative claim in [RESEARCH.md](../RESEARCH.md)
from raw data. Run it top-to-bottom to verify the reported numbers.

**Requirements:** `pip install -r ../requirements.txt` · Python 3.11+ · Free Alpaca IEX key (optional)

**Runtime:** ~8–12 minutes (50 tickers × walk-forward LightGBM)

---

## Table of Contents

1. [Setup & Data](#1-setup)
2. [Microstructure Feature Matrix](#2-features)
3. [Walk-Forward IC & Performance](#3-walkforward)
4. [Rank Fraction Sensitivity](#4-sensitivity)
5. [Two-Tier Signal Classification](#5-classification)
6. [Benjamini-Hochberg FDR](#6-fdr)
7. [Portfolio Simulation (net-of-cost)](#7-portfolio)
8. [Daily OFI Cross-Section](#8-daily)
9. [Summary Table (RESEARCH.md §4)](#9-summary)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, t as t_dist
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

from alpha_flow.config.settings import (
    TICKERS, SIGNAL_RANK_FRACTION, SIGNAL_SIGNIFICANCE_ALPHA,
    WF_TRAIN_WINDOW, WF_TEST_WINDOW, WF_HORIZON,
)
print(f'Universe: {len(TICKERS)} tickers')
print(f'Rank fraction: {SIGNAL_RANK_FRACTION} (quintile sort — Fama-French 1993)')
print(f'FDR target Q: {SIGNAL_SIGNIFICANCE_ALPHA}')
print(f'Walk-forward: train={WF_TRAIN_WINDOW}d ({WF_TRAIN_WINDOW*5}h), test={WF_TEST_WINDOW}d ({WF_TEST_WINDOW*5}h), horizon={WF_HORIZON}')

Universe: 50 tickers
Rank fraction: 0.2 (quintile sort — Fama-French 1993)
FDR target Q: 0.1
Walk-forward: train=252d (1260h), test=21d (105h), horizon=1


## 1. Setup & Data <a id='1-setup'></a>

Fetch 2-year hourly bars for all 50 tickers. Uses Alpaca IEX if keys are set,
falls back to yfinance hourly.

In [2]:
from alpha_flow.data.intraday_feed import get_intraday_bars

bars = {}
for t in TICKERS:
    df = get_intraday_bars(t, resolution='1h')
    bars[t] = df
    print(f'  {t}: {len(df)} bars  [{df.index[0].date()} → {df.index[-1].date()}]' if len(df) > 0 else f'  {t}: no data')

print(f'\nLoaded {len(bars)} tickers, median {np.median([len(v) for v in bars.values()]):.0f} bars')

  AAPL: 3721 bars  [2024-07-11 → 2026-07-10]
  MSFT: 3723 bars  [2024-07-11 → 2026-07-10]
  NVDA: 4118 bars  [2024-07-11 → 2026-07-10]
  META: 3645 bars  [2024-07-11 → 2026-07-10]
  GOOGL: 3767 bars  [2024-07-11 → 2026-07-10]
  AMZN: 3603 bars  [2024-07-11 → 2026-07-10]
  AVGO: 3777 bars  [2024-07-11 → 2026-07-10]
  ORCL: 3727 bars  [2024-07-11 → 2026-07-10]
  AMD: 3762 bars  [2024-07-11 → 2026-07-10]
  INTC: 3867 bars  [2024-07-11 → 2026-07-10]
  TSM: 3790 bars  [2024-07-11 → 2026-07-10]
  JPM: 3493 bars  [2024-07-11 → 2026-07-10]
  BAC: 3518 bars  [2024-07-11 → 2026-07-10]
  V: 3492 bars  [2024-07-11 → 2026-07-10]
  GS: 3477 bars  [2024-07-11 → 2026-07-10]
  WFC: 3495 bars  [2024-07-11 → 2026-07-10]
  MS: 3475 bars  [2024-07-11 → 2026-07-10]
  BLK: 3390 bars  [2024-07-11 → 2026-07-10]
  C: 3489 bars  [2024-07-11 → 2026-07-10]
  AXP: 3472 bars  [2024-07-11 → 2026-07-10]
  MA: 3482 bars  [2024-07-11 → 2026-07-10]
  JNJ: 3482 bars  [2024-07-11 → 2026-07-10]
  UNH: 3488 bars  [2024-07-11

## 2. Microstructure Feature Matrix <a id='2-features'></a>

Build the 13-feature matrix for one ticker (AAPL) to show feature statistics
and verify no look-ahead bias in construction.

In [3]:
from alpha_flow.analysis.intraday_engine import build_intraday_feature_matrix, FEATURE_COLS

feats_aapl = build_intraday_feature_matrix(bars['AAPL'])
print(f'AAPL feature matrix: {feats_aapl.shape[0]} rows × {len(FEATURE_COLS)} features + target')
print(f'Features: {FEATURE_COLS}')
print(f'Date range: {feats_aapl.index[0]} → {feats_aapl.index[-1]}')
print()
display(feats_aapl[FEATURE_COLS].describe().round(4))

AAPL feature matrix: 3690 rows × 13 features + target
Features: ['ofi_zscore', 'amihud', 'kyle_lambda', 'cs_spread', 'tick_sign', 'vwap_zscore', 'volume_zscore', 'hawkes_zscore', 'vpin_zscore', 'ret_1h', 'ret_3h', 'ret_6h', 'vol_ratio']
Date range: 2024-07-17 15:00:00+00:00 → 2026-07-10 18:00:00+00:00



,ofi_zscore,amihud,kyle_lambda,cs_spread,tick_sign,vwap_zscore,volume_zscore,hawkes_zscore,vpin_zscore,ret_1h,ret_3h,ret_6h,vol_ratio
count,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000,3690.0000
mean,0.0009,0.0048,-0.0153,0.0015,0.0179,-0.0208,0.0049,-0.0061,-0.0017,0.0001,0.0004,0.0007,1.0057
std,0.9695,0.0074,1.1186,0.0006,1.0000,1.0474,1.2764,1.1670,1.1688,0.0055,0.0100,0.0147,0.5903
min,-1.4888,0.0001,-2.6785,0.0005,-1.0000,-2.5590,-2.3992,-2.0601,-2.6200,-0.0171,-0.0324,-0.0503,0.0006
25%,-0.9747,0.0001,-0.7412,0.0011,-1.0000,-0.7314,-1.0917,-0.9245,-0.8591,-0.0025,-0.0046,-0.0067,0.6192
50%,0.6381,0.0014,-0.0957,0.0014,1.0000,0.0183,-0.0385,-0.1730,0.0048,0.0000,0.0004,0.0011,0.9121
75%,0.8816,0.0062,0.7237,0.0018,1.0000,0.6980,1.1111,0.8057,0.8195,0.0027,0.0053,0.0086,1.3066
max,1.3283,0.0403,2.9734,0.0038,1.0000,2.4302,2.5157,2.9226,2.6514,0.0192,0.0319,0.0424,2.9641


### Feature correlation matrix

Low inter-feature correlation means each signal contributes independent information
to the LightGBM model (desirable — highly correlated features waste splits).

In [4]:
corr = feats_aapl[FEATURE_COLS].corr(method='spearman')
# Show pairs with |ρ| > 0.5 (potential redundancy)
high_corr = []
for i in range(len(FEATURE_COLS)):
    for j in range(i+1, len(FEATURE_COLS)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            high_corr.append((FEATURE_COLS[i], FEATURE_COLS[j], round(r, 3)))
if high_corr:
    print('Feature pairs with |Spearman ρ| > 0.5:')
    for a, b, r in high_corr:
        print(f'  {a} × {b}: {r:+.3f}')
else:
    print('No feature pairs with |ρ| > 0.5 — good orthogonality.')
print(f'\nMean off-diagonal |ρ|: {np.abs(corr.values[np.triu_indices(len(FEATURE_COLS), k=1)]).mean():.3f}')

Feature pairs with |Spearman ρ| > 0.5:
  ofi_zscore × tick_sign: +0.731
  tick_sign × vwap_zscore: +0.508
  ret_1h × ret_3h: +0.525
  ret_3h × ret_6h: +0.641

Mean off-diagonal |ρ|: 0.097


## 3. Walk-Forward IC & Performance <a id='3-walkforward'></a>

Run the full walk-forward LightGBM pipeline on all 50 tickers.
This is the core of RESEARCH.md §4.1 — every number below should match.

In [5]:
from alpha_flow.analysis.intraday_engine import run_intraday_pipeline

results = run_intraday_pipeline(TICKERS, resolution='1h')
print(f'Pipeline complete: {len(results)} tickers')
errors = {t: r.get('error') for t, r in results.items() if 'error' in r}
if errors:
    print(f'Errors ({len(errors)}): {errors}')

/Users/anto/Scholarship/Projects/AlphaFlow/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline complete: 50 tickers


In [6]:
# Build results table — this is Table 1 in RESEARCH.md §4.1
valid = {t: r for t, r in results.items() if 'error' not in r}

rows = []
for t, r in sorted(valid.items()):
    rows.append({
        'Ticker': t,
        'IC (%)': round(r['mean_ic'] * 100, 2),
        'IC SEM': round(r.get('ic_sem', 0) * 100, 2),
        'IC t-stat': round(r.get('ic_tstat', 0), 2),
        'IC p-value': round(r.get('ic_pvalue', 1), 4),
        'IC_IR': round(r.get('ic_ir', 0), 2),
        'Sharpe': round(r.get('sharpe', 0), 2),
        'Sortino': round(r.get('sortino', 0), 2),
        'Hit Rate': round(r.get('hit_rate', 0) * 100, 1),
        'Max DD (%)': round(r.get('max_drawdown', 0) * 100, 1),
        'Folds': r.get('n_folds', 0),
        'Latest Signal': round(r.get('latest_signal', 0), 6),
    })

df_results = pd.DataFrame(rows).set_index('Ticker')
display(df_results)

# Cross-sectional summary statistics (RESEARCH.md §4.1 headline numbers)
ics = df_results['IC (%)'].values
print(f'\n--- Cross-sectional summary (N={len(valid)} tickers) ---')
print(f'Avg |IC|:      {np.mean(np.abs(ics)):.2f}%')
print(f'Median |IC|:   {np.median(np.abs(ics)):.2f}%')
print(f'Max |IC|:      {np.max(np.abs(ics)):.2f}% ({df_results["IC (%)"].abs().idxmax()})')
print(f'Avg Sharpe:    {df_results["Sharpe"].mean():.2f}')
print(f'Avg Sortino:   {df_results["Sortino"].mean():.2f}')
print(f'Avg Hit Rate:  {df_results["Hit Rate"].mean():.1f}%')
print(f'Folds range:   {df_results["Folds"].min()}–{df_results["Folds"].max()}')

,IC (%),IC SEM,IC t-stat,IC p-value,IC_IR,Sharpe,Sortino,Hit Rate,Max DD (%),Folds,Latest Signal
Ticker,,,,,,,,,,,
AAPL,-2.6300,1.7600,-1.5000,0.1485,-1.5000,0.2100,0.3100,50.8000,-22.1000,23,-0.0028
ABBV,-2.0500,2.4100,-0.8500,0.4036,-0.8500,0.2200,0.3200,49.5000,-12.0000,20,0.0005
AMD,1.9800,2.1800,0.9100,0.3741,0.9100,1.4500,2.0000,51.1000,-9.9000,23,0.0007
AMZN,-0.3500,2.0900,-0.1700,0.8672,-0.1700,-0.6200,-0.7600,51.3000,-18.4000,22,-0.0009
AVGO,1.1400,2.2600,0.5000,0.6198,0.5000,0.4400,0.5900,51.8000,-13.3000,23,-0.0014
AXP,-0.0800,2.0100,-0.0400,0.9703,-0.0400,1.2800,2.1900,50.3000,-10.1000,20,-0.0053
BA,1.5800,2.3400,0.6800,0.5071,0.6800,1.6000,2.3500,51.8000,-14.4000,20,0.0032
BAC,-0.7700,2.2900,-0.3300,0.7419,-0.3300,0.7800,0.9600,50.8000,-14.6000,21,-0.0004
BLK,-4.0200,2.2300,-1.8000,0.0881,-1.8000,1.2900,1.8300,51.9000,-10.5000,19,-0.0010



--- Cross-sectional summary (N=50 tickers) ---
Avg |IC|:      1.71%
Median |IC|:   1.49%
Max |IC|:      5.08% (TSM)
Avg Sharpe:    0.54
Avg Sortino:   0.78
Avg Hit Rate:  50.4%
Folds range:   19–26


## 4. Rank Fraction Sensitivity <a id='4-sensitivity'></a>

How does the long-short book perform at different rank fractions?
This justifies `SIGNAL_RANK_FRACTION = 0.20` (quintile sort).

**What to look for:** the book Sharpe should peak or plateau around 0.15–0.25,
degrading at extremes (0.05 = too concentrated, 0.40 = too diluted).

In [7]:
from alpha_flow.analysis.signal_classification import classify_signal

fractions = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
sensitivity_rows = []

sorted_by_signal = sorted(valid.keys(), key=lambda t: valid[t].get('latest_signal', 0), reverse=True)
n = len(sorted_by_signal)

for frac in fractions:
    n_leg = max(1, round(n * frac))
    buy_set = set(sorted_by_signal[:n_leg])
    sell_set = set(sorted_by_signal[n - n_leg:])
    
    buys, sells, holds = 0, 0, 0
    buy_ics, sell_ics = [], []
    for t in valid:
        ls = valid[t].get('latest_signal', 0)
        sig = classify_signal(
            signal_value=ls, in_buy_rank=t in buy_set, in_sell_rank=t in sell_set,
            sign_ok_buy=(ls >= 0), sign_ok_sell=(ls <= 0), abs_threshold=float('inf'),
        )
        if sig == 'BUY': buys += 1; buy_ics.append(valid[t]['mean_ic'])
        elif sig == 'SELL': sells += 1; sell_ics.append(valid[t]['mean_ic'])
        else: holds += 1
    
    avg_long_ic = np.mean(buy_ics) * 100 if buy_ics else 0
    avg_short_ic = np.mean(np.abs(sell_ics)) * 100 if sell_ics else 0
    spread_ic = avg_long_ic + avg_short_ic
    
    sensitivity_rows.append({
        'Fraction': f'{frac:.0%}',
        'Long': buys,
        'Short': sells,
        'Hold': holds,
        'Avg Long IC (%)': round(avg_long_ic, 2),
        'Avg |Short IC| (%)': round(avg_short_ic, 2),
        'IC Spread (%)': round(spread_ic, 2),
    })

df_sens = pd.DataFrame(sensitivity_rows).set_index('Fraction')
display(df_sens)
print('\nIC Spread = Avg Long IC + Avg |Short IC| — the cross-sectional edge the book monetises.')
print('Higher spread = more profitable rank sort (before transaction costs).')
print(f'Current setting: SIGNAL_RANK_FRACTION = {SIGNAL_RANK_FRACTION}')

,Long,Short,Hold,Avg Long IC (%),Avg |Short IC| (%),IC Spread (%)
Fraction,,,,,,
10%,5,5,40,2.1000,1.0800,3.1800
15%,8,8,34,1.6300,1.1300,2.7600
20%,10,10,30,1.2700,1.0500,2.3200
25%,12,12,26,1.6900,1.2900,2.9700
30%,15,15,20,1.1500,1.5100,2.6600
40%,20,20,10,0.7600,1.4800,2.2400



IC Spread = Avg Long IC + Avg |Short IC| — the cross-sectional edge the book monetises.
Higher spread = more profitable rank sort (before transaction costs).
Current setting: SIGNAL_RANK_FRACTION = 0.2


## 5. Two-Tier Signal Classification <a id='5-classification'></a>

Demonstrate the core design: Tier-1 (tradeable book, rank-based, no FDR gate)
and Tier-2 (high-conviction flag, BH-FDR, annotation only).

In [8]:
from alpha_flow.analysis.signal_classification import (
    classify_signal, is_high_conviction, benjamini_hochberg_threshold,
)

# Tier-1: rank by latest_signal, top/bottom 20%
n_candidates = max(1, round(n * SIGNAL_RANK_FRACTION))
buy_rank = set(sorted_by_signal[:n_candidates])
sell_rank = set(sorted_by_signal[n - n_candidates:])

# Tier-2: BH-FDR across all p-values
all_pvalues = [valid[t].get('ic_pvalue', 1.0) for t in valid]
fdr_thr = benjamini_hochberg_threshold(all_pvalues, SIGNAL_SIGNIFICANCE_ALPHA)
print(f'BH-FDR threshold at Q={SIGNAL_SIGNIFICANCE_ALPHA}: {fdr_thr:.6f}')
print(f'Best p-value in batch: {min(all_pvalues):.6f}')
print(f'Needed for rank 1 of {n}: p ≤ (1/{n})×{SIGNAL_SIGNIFICANCE_ALPHA} = {SIGNAL_SIGNIFICANCE_ALPHA/n:.6f}')
print()

classification_rows = []
for t in sorted_by_signal:
    r = valid[t]
    ls = r.get('latest_signal', 0)
    pval = r.get('ic_pvalue', 1.0)
    sig = classify_signal(
        signal_value=ls, in_buy_rank=t in buy_rank, in_sell_rank=t in sell_rank,
        sign_ok_buy=(ls >= 0), sign_ok_sell=(ls <= 0), abs_threshold=float('inf'),
    )
    hc = is_high_conviction(pval, fdr_thr)
    classification_rows.append({
        'Ticker': t, 'Latest Signal': round(ls, 6),
        'IC (%)': round(r['mean_ic'] * 100, 2),
        'p-value': round(pval, 4),
        'Tier-1 (Book)': sig,
        'Tier-2 (Conviction)': 'HIGH' if hc else '-',
    })

df_class = pd.DataFrame(classification_rows).set_index('Ticker')
display(df_class)

buys = (df_class['Tier-1 (Book)'] == 'BUY').sum()
sells = (df_class['Tier-1 (Book)'] == 'SELL').sum()
holds = (df_class['Tier-1 (Book)'] == 'HOLD').sum()
hc_count = (df_class['Tier-2 (Conviction)'] == 'HIGH').sum()
print(f'\nBook: {buys} BUY / {sells} SELL / {holds} HOLD')
print(f'High-conviction (FDR): {hc_count} of {n}')

BH-FDR threshold at Q=0.1: 0.000000
Best p-value in batch: 0.007110
Needed for rank 1 of 50: p ≤ (1/50)×0.1 = 0.002000



,Latest Signal,IC (%),p-value,Tier-1 (Book),Tier-2 (Conviction)
Ticker,,,,,
TSM,0.0139,5.0800,0.0331,BUY,-
UNH,0.0060,1.6100,0.3790,BUY,-
INTC,0.0048,-0.6300,0.7264,BUY,-
PFE,0.0041,1.9300,0.4582,BUY,-
HON,0.0032,2.5200,0.3068,BUY,-
BA,0.0032,1.5800,0.5071,BUY,-
MS,0.0030,2.6000,0.4906,BUY,-
TMO,0.0025,-1.6400,0.3549,BUY,-
COP,0.0019,1.0800,0.6795,BUY,-



Book: 10 BUY / 10 SELL / 30 HOLD
High-conviction (FDR): 0 of 50


## 6. Benjamini-Hochberg FDR — Worked Example <a id='6-fdr'></a>

Why 0 names survive FDR correction on free data:

- 50 simultaneous tests, Q = 0.10
- For the best p-value (rank 1) to survive: p₁ ≤ (1/50) × 0.10 = 0.002
- Free OHLCV hourly data produces IC ≈ 1.4% → typical p-values ≈ 0.3–0.8
- Even the best p-value (typically ≈ 0.01–0.05) exceeds 0.002

This is the **correct** statistical answer. The book still trades (Tier-1 is not gated).

In [9]:
sorted_pvals = sorted(all_pvalues)
print(f'{"Rank":>4}  {"p-value":>10}  {"BH threshold":>14}  {"Survives?":>10}')
print('-' * 48)
for k, p in enumerate(sorted_pvals[:10], start=1):
    bh_k = (k / len(sorted_pvals)) * SIGNAL_SIGNIFICANCE_ALPHA
    survives = '  YES' if p <= bh_k else '  no'
    print(f'{k:>4}  {p:>10.6f}  {bh_k:>14.6f}  {survives:>10}')
print(f'  ...')
print(f'\nResult: {sum(1 for p in sorted_pvals if p <= fdr_thr)} names survive at Q={SIGNAL_SIGNIFICANCE_ALPHA}')

Rank     p-value    BH threshold   Survives?
------------------------------------------------
   1    0.007110        0.002000          no
   2    0.033106        0.004000          no
   3    0.038459        0.006000          no
   4    0.042174        0.008000          no
   5    0.088093        0.010000          no
   6    0.137852        0.012000          no
   7    0.140291        0.014000          no
   8    0.148475        0.016000          no
   9    0.169890        0.018000          no
  10    0.206663        0.020000          no
  ...

Result: 0 names survive at Q=0.1


## 7. Portfolio Simulation (net-of-cost) <a id='7-portfolio'></a>

Cross-sectional long-short portfolio: long top-3 IC, short bottom-3 IC,
with Corwin-Schultz half-spread transaction costs at monthly rebalance.

In [10]:
from alpha_flow.analysis.portfolio_engine import build_longshort_portfolio

cards = []
for t, r in valid.items():
    if r.get('equity_curve') and len(r['equity_curve']) > 10:
        cards.append({'ticker': t, **r})

port = build_longshort_portfolio(cards, n_long=3, n_short=3)

if 'error' not in port:
    print(f'Long:  {port["long_tickers"]}')
    print(f'Short: {port["short_tickers"]}')
    print(f'Gross Sharpe:    {port["gross_sharpe"]:+.4f}')
    print(f'Net Sharpe:      {port["net_sharpe"]:+.4f}')
    print(f'Avg cost (bps):  {port["avg_cost_bps"]:.1f}')
    print(f'Net Max DD:      {port["net_max_drawdown"]:.2%}')
    print(f'Hit Rate:        {port["hit_rate"]:.1%}')
    print(f'Profit Factor:   {port["profit_factor"]:.2f}')
    print(f'Rebalances:      {port["n_rebalances"]}')
    print(f'Bars:            {port["n_bars"]}')
else:
    print(f'Portfolio error: {port["error"]}')

Long:  ['TSM', 'KO', 'NFLX']
Short: ['WMT', 'BLK', 'ORCL']
Gross Sharpe:    -0.1733
Net Sharpe:      -0.3726
Avg cost (bps):  16.7
Net Max DD:      -16.41%
Hit Rate:        49.1%
Profit Factor:   0.99
Rebalances:      18
Bars:            1994


## 8. Daily OFI Cross-Section <a id='8-daily'></a>

Daily OFI IC ≈ 0 is the expected result: Chordia et al. (2002) measured OFI
on TAQ tick data, not daily OHLCV bars. The OFI signal's half-life is ~30 min,
so daily bars average it out to noise.

In [11]:
from alpha_flow.data.data_feed import get_daily_bars
from alpha_flow.core.ofi_calculator import rolling_ofi_zscore

daily_ics = []
for t in TICKERS:
    df = get_daily_bars(t, years=2)
    if len(df) < 50:
        continue
    ofi_z = rolling_ofi_zscore(df)
    fwd = df['close'].pct_change().shift(-1)
    common = ofi_z.dropna().index.intersection(fwd.dropna().index)
    if len(common) >= 20:
        ic, _ = spearmanr(ofi_z.loc[common], fwd.loc[common])
        if not np.isnan(ic):
            daily_ics.append((t, ic))

daily_ics_arr = np.array([x[1] for x in daily_ics])
print(f'Daily OFI IC across {len(daily_ics)} tickers:')
print(f'  Mean IC:   {np.mean(daily_ics_arr):+.4f}')
print(f'  Median IC: {np.median(daily_ics_arr):+.4f}')
print(f'  Std IC:    {np.std(daily_ics_arr):.4f}')
print(f'\nExpected: ≈ 0 on daily OHLCV (OFI half-life ~30min, daily bars average it out).')

  [clean/AVGO] Clipping 1 extreme return rows (>±20%)
  [clean/AVGO] Total: 1/501 rows cleaned (0.2%). Remaining: 500
  [clean/ORCL] Clipping 1 extreme return rows (>±20%)
  [clean/ORCL] Total: 1/501 rows cleaned (0.2%). Remaining: 500
  [clean/AMD] Clipping 2 extreme return rows (>±20%)
  [clean/AMD] Total: 2/501 rows cleaned (0.4%). Remaining: 499
  [clean/INTC] Clipping 3 extreme return rows (>±20%)
  [clean/INTC] Total: 3/501 rows cleaned (0.6%). Remaining: 498
  [clean/UNH] Clipping 1 extreme return rows (>±20%)
  [clean/UNH] Total: 1/501 rows cleaned (0.2%). Remaining: 500
  [clean/TSLA] Clipping 2 extreme return rows (>±20%)
  [clean/TSLA] Total: 2/501 rows cleaned (0.4%). Remaining: 499
  [clean/SBUX] Clipping 1 extreme return rows (>±20%)
  [clean/SBUX] Total: 1/501 rows cleaned (0.2%). Remaining: 500
Daily OFI IC across 50 tickers:
  Mean IC:   -0.0074
  Median IC: -0.0114
  Std IC:    0.0406

Expected: ≈ 0 on daily OHLCV (OFI half-life ~30min, daily bars average it out).


## 9. Summary Table <a id='9-summary'></a>

This is the single source of truth. Every number in RESEARCH.md and README.md
should match this table exactly.

In [12]:
summary = {
    'Universe size': len(TICKERS),
    'Features': len(FEATURE_COLS),
    'Walk-forward folds (range)': f'{df_results["Folds"].min()}–{df_results["Folds"].max()}',
    'Avg |IC| (%)': round(np.mean(np.abs(ics)), 2),
    'Median |IC| (%)': round(np.median(np.abs(ics)), 2),
    'Max |IC| (%)': f'{np.max(np.abs(ics)):.2f} ({df_results["IC (%)"].abs().idxmax()})',
    'Avg Sharpe': round(df_results['Sharpe'].mean(), 2),
    'Avg Sortino': round(df_results['Sortino'].mean(), 2),
    'Book (20% quintile)': f'{buys}L / {sells}S / {holds}H',
    'High-conviction (FDR)': f'{hc_count} of {n}',
    'Rank fraction': SIGNAL_RANK_FRACTION,
    'FDR target Q': SIGNAL_SIGNIFICANCE_ALPHA,
    'Daily OFI IC': f'{np.mean(daily_ics_arr):+.4f} (expected ≈ 0)',
}

for k, v in summary.items():
    print(f'{k:.<35} {v}')

print('\n--- Verify against RESEARCH.md §4 and README.md ---')
print('If any number differs, update the docs to match this notebook.')

Universe size...................... 50
Features........................... 13
Walk-forward folds (range)......... 19–26
Avg |IC| (%)....................... 1.71
Median |IC| (%).................... 1.49
Max |IC| (%)....................... 5.08 (TSM)
Avg Sharpe......................... 0.54
Avg Sortino........................ 0.78
Book (20% quintile)................ 10L / 10S / 30H
High-conviction (FDR).............. 0 of 50
Rank fraction...................... 0.2
FDR target Q....................... 0.1
Daily OFI IC....................... -0.0074 (expected ≈ 0)

--- Verify against RESEARCH.md §4 and README.md ---
If any number differs, update the docs to match this notebook.


---

## Appendix: SHAP Feature Importance (cross-sectional)

Which features matter most across all 50 tickers?

In [13]:
# Aggregate SHAP importance across all tickers
shap_agg = {f: 0.0 for f in FEATURE_COLS}
count = 0
for t, r in valid.items():
    shap = r.get('shap_importance', {})
    if shap:
        for f in FEATURE_COLS:
            shap_agg[f] += shap.get(f, 0)
        count += 1

if count > 0:
    shap_agg = {f: v / count for f, v in shap_agg.items()}
    shap_sorted = sorted(shap_agg.items(), key=lambda x: x[1], reverse=True)
    print(f'Mean |SHAP| across {count} tickers (last fold):')
    for f, v in shap_sorted:
        bar = '█' * int(v / max(shap_agg.values()) * 30)
        print(f'  {f:<18} {v:.6f}  {bar}')

Mean |SHAP| across 50 tickers (last fold):
  vol_ratio          0.000659  ██████████████████████████████
  ret_6h             0.000491  ██████████████████████
  ret_1h             0.000485  ██████████████████████
  cs_spread          0.000482  █████████████████████
  amihud             0.000474  █████████████████████
  vwap_zscore        0.000456  ████████████████████
  hawkes_zscore      0.000450  ████████████████████
  ret_3h             0.000443  ████████████████████
  volume_zscore      0.000412  ██████████████████
  kyle_lambda        0.000395  █████████████████
  vpin_zscore        0.000377  █████████████████
  ofi_zscore         0.000304  █████████████
  tick_sign          0.000093  ████
